# Aula 11 – Prática: o LCM em Python
### Detecção de mudança, potencial de transição e predição – Bacia Pirajibu-Mirim

Este notebook refaz, passo a passo, o que o **Land Change Modeler (LCM)** do TerrSet faz, usando Python no Colab.
Ele lê **os mesmos arquivos `.rst`/`.rdc`** usados no TerrSet, então os resultados podem ser comparados lado a lado.

| Etapa | Aba do LCM | Seção do notebook |
|---|---|---|
| Ganhos, perdas, mudança líquida, troca, contribuições | Change Analysis | 2 |
| Mapa de transições e tendência espacial | Change Maps / Spatial Trend of Change | 3 |
| Seleção de variáveis (V de Cramér) | Transition Potentials – Test and Selection | 4 |
| Potencial de transição com rede neural (MLP) | Transition Potentials – Run Transition Sub-Model | 5 |
| Quantidade de mudança (Cadeia de Markov) | Change Prediction – Change Demand Modeling | 6 |
| Alocação: predição *hard* e *soft* | Change Prediction | 7 |
| Validação (figura de mérito) | Validation | 8 |
| Cenário futuro | Change Prediction (recalibrado) | 9 |

**Desenho do experimento**

- **Calibração:** T1 = 1995 → T2 = 2010
- **Validação:** prever 2025 a partir de 2010 e comparar com o 2025 real
- **Cenário:** recalibrar com 2010 → 2025 e projetar 2040

> A diferença em relação ao TerrSet é que aqui **cada conta fica visível**: dá para abrir a "caixa-preta" do LCM.

## 1. Configuração e leitura dos dados

Coloque a pasta do TerrSet (`LCM-posIA`) no seu Google Drive. O notebook procura os arquivos `.rst` **em todas as subpastas**,
então não importa se os mapas estão em `1_Land_cover` e as variáveis em outra pasta.

Arquivos esperados: `uso_1995`, `uso_2010`, `uso_2025` e as variáveis listadas em `VARIAVEIS` (cada um com o seu `.rdc`).

In [ ]:
# Conectar ao Google Drive (no Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: usando a pasta local definida em PASTA.')

In [ ]:
#CONFIGURAÇÃO – ajuste aqui
PASTA = '/content/drive/MyDrive/LCM-posIA'   # pasta com os .rst/.rdc (procura em subpastas)

ANO_T1, ANO_T2, ANO_T3 = 1995, 2010, 2025     # calibração 1995→2010, validação em 2025
ANO_CENARIO = 2040                            # cenário: 2010→2025 projetado para 2040

# Legenda (MapBiomas nível 1) e cores
CLASSES = {1: 'Floresta', 10: 'Formação natural não florestal', 14: 'Agropecuária',
           22: 'Área não vegetada', 26: "Corpo d'água"}
CORES = {1: '#1f8d49', 10: '#d6bc74', 14: '#ffefc3', 22: '#d4271e', 26: '#2532e4'}
CLASSE_FOCO = 22                              # classe analisada em detalhe (urbanização)

# Variáveis explicativas (nome do arquivo sem .rst). As que não existirem são ignoradas.
VARIAVEIS = ['d_city', 'd_water', 'd_energy', 'd_road', 'mde', 'slope', 'rusle']

# Variável DINÂMICA: recalculada a partir do mapa a cada etapa da simulação
# (distância até a classe 22). Use None para tratar todas como estáticas.
VAR_DINAMICA = 'd_city'
CLASSE_DINAMICA = 22

# Submodelos: cada um é um grupo de transições (de, para) explicadas pelas mesmas variáveis.
# Ajuste depois de olhar a matriz de transição da Seção 2.
SUBMODELOS = {
    'Expansão urbana':     [(14, 22), (1, 22), (10, 22)],
    'Supressão florestal': [(1, 14)],
    'Regeneração':         [(14, 1)],
}

AMOSTRAS_POR_CLASSE = 3000   # máximo de pixels por classe no treino da MLP (como no LCM)
N_ETAPAS = 3                 # etapas de recálculo da variável dinâmica na predição
SEMENTE = 42

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy import ndimage
from scipy.linalg import fractional_matrix_power
from scipy.stats import chi2_contingency
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.float_format', lambda v: f'{v:,.1f}')
rng = np.random.default_rng(SEMENTE)
COD = list(CLASSES)                     # códigos das classes, em ordem

### Leitura dos arquivos Idrisi (`.rst` + `.rdc`)

O `.rst` é só uma sequência de números (um por pixel, linha por linha). O `.rdc` é um texto que diz
quantas linhas e colunas existem, o tipo de dado e as coordenadas. Por isso dá para ler com o NumPy, sem GDAL.

In [ ]:
TIPOS = {'byte': np.uint8, 'integer': np.int16, 'real': np.float32}

def ler_rdc(caminho):
    meta, legenda = {}, {}
    for linha in open(caminho, encoding='latin-1'):
        if ':' not in linha:
            continue
        chave, valor = linha.split(':', 1)
        chave, valor = chave.strip(), valor.strip()
        if chave.startswith('code'):
            legenda[int(chave.split()[1])] = valor
        else:
            meta[chave] = valor
    meta['legenda'] = legenda
    meta['arquivo'] = caminho
    return meta

def achar(nome):
    achados = sorted(glob.glob(os.path.join(PASTA, '**', nome + '.rst'), recursive=True))
    return achados[0] if achados else None

def ler_rst(nome):
    rst = achar(nome)
    if rst is None:
        raise FileNotFoundError(f'{nome}.rst não encontrado em {PASTA}')
    meta = ler_rdc(rst[:-4] + '.rdc')
    linhas, colunas = int(meta['rows']), int(meta['columns'])
    if meta.get('file type', 'binary') != 'binary':
        raise ValueError(f'{nome}: só o formato binário é suportado (file type = {meta["file type"]})')
    dados = np.fromfile(rst, dtype=TIPOS[meta['data type']])
    if dados.size != linhas * colunas:
        raise ValueError(f'{nome}: o .rst tem {dados.size} valores, mas o .rdc indica {linhas}×{colunas}')
    return dados.reshape(linhas, colunas), meta

In [ ]:
# Mapas de uso e cobertura
uso, meta_uso = {}, {}
for ano in (ANO_T1, ANO_T2, ANO_T3):
    uso[ano], meta_uso[ano] = ler_rst(f'uso_{ano}')
    print(f'uso_{ano}: {uso[ano].shape[0]} linhas × {uso[ano].shape[1]} colunas | valores: {np.unique(uso[ano]).tolist()}')

formas = {a.shape for a in uso.values()}
assert len(formas) == 1, 'Os mapas de uso têm tamanhos diferentes!'
LINHAS, COLUNAS = uso[ANO_T1].shape

# Área de estudo: pixels com classe válida nos três anos
mascara = np.all([np.isin(uso[a], COD) for a in uso], axis=0)

# Tamanho do pixel em metros (o .rdc pode estar em graus ou em metros)
m = meta_uso[ANO_T1]
xmin, xmax, ymin, ymax = (float(m[k]) for k in ('min. X', 'max. X', 'min. Y', 'max. Y'))
if m.get('ref. units', 'm').startswith('deg'):
    lat_media = np.radians((ymin + ymax) / 2)
    DX = (xmax - xmin) / COLUNAS * 111320 * np.cos(lat_media)
    DY = (ymax - ymin) / LINHAS * 110574
else:
    DX, DY = (xmax - xmin) / COLUNAS, (ymax - ymin) / LINHAS
AREA_HA = DX * DY / 1e4
print(f'\nPixel ≈ {DX:.1f} m × {DY:.1f} m = {AREA_HA:.4f} ha')
print(f'Área de estudo: {mascara.sum():,} pixels = {mascara.sum() * AREA_HA:,.0f} ha')

In [ ]:
# Variáveis explicativas
def limpar(nome, arr, meta):
    a = arr.astype(np.float64)
    a[~np.isfinite(a) | (np.abs(a) > 1e30)] = np.nan          # ex.: -3.4e38 do rusle
    try:
        flag = float(meta.get('flag value', 'none'))
        if not (flag == 0 and nome.startswith('d_')):          # distância 0 é um valor válido
            a[a == flag] = np.nan
    except ValueError:
        pass
    lacunas = np.isnan(a) & mascara
    if lacunas.any():
        mediana = np.nanmedian(a[mascara])
        print(f'  {nome}: {lacunas.sum()} pixels sem dado na área de estudo → preenchidos com a mediana ({mediana:.2f})')
        a[np.isnan(a)] = mediana
    a[np.isnan(a)] = 0
    return a

variaveis = {}
for nome in VARIAVEIS:
    if achar(nome) is None:
        print(f'  {nome}: arquivo não encontrado – ignorada')
        continue
    arr, meta = ler_rst(nome)
    if arr.shape != (LINHAS, COLUNAS):
        print(f'  {nome}: tamanho {arr.shape} diferente dos mapas – ignorada')
        continue
    variaveis[nome] = limpar(nome, arr, meta)

NOMES_VAR = list(variaveis)
pd.DataFrame({n: [variaveis[n][mascara].min(), variaveis[n][mascara].mean(), variaveis[n][mascara].max()]
              for n in NOMES_VAR}, index=['mín', 'média', 'máx']).T

In [ ]:
# Funções de visualização
# Os códigos (1, 10, 14, 22, 26) viram posições 1..5 de uma paleta; 0 = fundo branco
LUT = np.zeros(256, dtype=np.int16)
for i, c in enumerate(COD, start=1):
    LUT[c] = i
CMAP_USO = ListedColormap(['#ffffff'] + [CORES[c] for c in COD])
LEGENDA_USO = [Patch(color=CORES[c], label=f'{c} – {CLASSES[c]}') for c in COD]

def mostrar_uso(mapa, titulo, ax):
    ax.imshow(np.where(mascara, LUT[mapa], 0), cmap=CMAP_USO, vmin=-0.5, vmax=len(COD) + 0.5,
              interpolation='nearest')
    ax.set_title(titulo); ax.axis('off')

def mostrar_continuo(arr, titulo, ax, cmap='viridis', vmin=None, vmax=None):
    im = ax.imshow(np.where(mascara, arr, np.nan), cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(titulo); ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.7)

fig, axs = plt.subplots(1, 3, figsize=(15, 7))
for ax, ano in zip(axs, uso):
    mostrar_uso(uso[ano], f'Uso e cobertura {ano}', ax)
fig.legend(handles=LEGENDA_USO, loc='lower center', ncol=5, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

## 2. Change Analysis: ganhos, perdas, mudança líquida e troca

Tudo parte da **matriz de transição**: linhas = classe em T1, colunas = classe em T2, valores em hectares.

- **Perda** de uma classe = soma da sua linha fora da diagonal
- **Ganho** = soma da sua coluna fora da diagonal
- **Mudança líquida** = ganho − perda
- **Troca (swap)** = 2 × mín(ganho, perda): a classe perde num lugar e ganha em outro
- **Mudança total** = ganho + perda = |líquida| + troca

In [ ]:
def matriz_transicao(t1, t2, unidade='ha'):
    a, b = t1[mascara], t2[mascara]
    m = pd.DataFrame([[np.sum((a == i) & (b == j)) for j in COD] for i in COD],
                     index=[f'{c} {CLASSES[c]}' for c in COD],
                     columns=[f'{c} {CLASSES[c]}' for c in COD], dtype=float)
    return m * AREA_HA if unidade == 'ha' else m

def ganhos_perdas(mt):
    v = mt.values
    tab = pd.DataFrame(index=mt.index)
    tab['Área T1'] = v.sum(axis=1)
    tab['Área T2'] = v.sum(axis=0)
    tab['Persistência'] = np.diag(v)
    tab['Perda'] = tab['Área T1'] - tab['Persistência']
    tab['Ganho'] = tab['Área T2'] - tab['Persistência']
    tab['Líquida'] = tab['Ganho'] - tab['Perda']
    tab['Troca'] = 2 * np.minimum(tab['Ganho'], tab['Perda'])
    tab['Total'] = tab['Ganho'] + tab['Perda']
    return tab

MT_CAL = matriz_transicao(uso[ANO_T1], uso[ANO_T2])
print(f'Matriz de transição {ANO_T1} → {ANO_T2} (ha) – linhas: {ANO_T1}, colunas: {ANO_T2}')
display(MT_CAL)
GP_CAL = ganhos_perdas(MT_CAL)
display(GP_CAL)

In [ ]:
# Gráficos como os do LCM: ganhos e perdas por classe e mudança líquida
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
y = np.arange(len(COD))
nomes = [CLASSES[c] for c in COD]

axs[0].barh(y, -GP_CAL['Perda'], color='#c46a2b', label='Perda')
axs[0].barh(y, GP_CAL['Ganho'], color='#2f6f9f', label='Ganho')
axs[0].set_yticks(y, nomes); axs[0].axvline(0, color='k', lw=0.8)
axs[0].set_title(f'Ganhos e perdas por classe ({ANO_T1}–{ANO_T2})'); axs[0].set_xlabel('ha'); axs[0].legend()

cores_liq = ['#2f6f9f' if v >= 0 else '#c46a2b' for v in GP_CAL['Líquida']]
axs[1].barh(y, GP_CAL['Líquida'], color=cores_liq)
axs[1].set_yticks(y, nomes); axs[1].axvline(0, color='k', lw=0.8)
axs[1].set_title('Mudança líquida por classe'); axs[1].set_xlabel('ha')
plt.tight_layout(); plt.show()

# Mudança líquida × troca: a líquida esconde a dinâmica?
GP_CAL[['Líquida', 'Troca', 'Total']].abs().rename(columns={'Líquida': '|Líquida|'}).plot.barh(
    figsize=(9, 4), color=['#5b6e76', '#e8a33d', '#0e3440'], title='|Mudança líquida|, troca e mudança total (ha)')
plt.gca().set_yticklabels(nomes); plt.tight_layout(); plt.show()

In [ ]:
# Contribuições para a mudança líquida da classe foco
i_foco = COD.index(CLASSE_FOCO)
v = MT_CAL.values
contrib = pd.Series({CLASSES[c]: v[COD.index(c), i_foco] - v[i_foco, COD.index(c)]
                     for c in COD if c != CLASSE_FOCO})
print(f'Contribuições para a mudança líquida de "{CLASSES[CLASSE_FOCO]}" (ha):')
display(contrib.to_frame('ha'))
contrib.plot.barh(color=['#2f6f9f' if x >= 0 else '#c46a2b' for x in contrib], figsize=(8, 3.5),
                  title=f'Contribuições para a mudança líquida de {CLASSES[CLASSE_FOCO]} ({ANO_T1}–{ANO_T2})')
plt.axvline(0, color='k', lw=0.8); plt.xlabel('ha'); plt.tight_layout(); plt.show()

# Taxa anual de mudança: r = ln(A2/A1) / (t2 - t1)
taxa = np.log(GP_CAL['Área T2'] / GP_CAL['Área T1'].replace(0, np.nan)) / (ANO_T2 - ANO_T1) * 100
print('Taxa anual de mudança (% ao ano):')
display(taxa.to_frame('% a.a.'))

### Tendência temporal: o ritmo se manteve?

Comparamos os dois períodos (1995–2010 e 2010–2025). Se as taxas mudam muito, a premissa da Cadeia de Markov
(o futuro repete o passado) fica frágil.

In [ ]:
GP_VAL = ganhos_perdas(matriz_transicao(uso[ANO_T2], uso[ANO_T3]))
comparacao = pd.DataFrame({
    f'Líquida {ANO_T1}–{ANO_T2}': GP_CAL['Líquida'],
    f'Líquida {ANO_T2}–{ANO_T3}': GP_VAL['Líquida'],
    f'Taxa % a.a. {ANO_T1}–{ANO_T2}': np.log(GP_CAL['Área T2'] / GP_CAL['Área T1'].replace(0, np.nan)) / (ANO_T2 - ANO_T1) * 100,
    f'Taxa % a.a. {ANO_T2}–{ANO_T3}': np.log(GP_VAL['Área T2'] / GP_VAL['Área T1'].replace(0, np.nan)) / (ANO_T3 - ANO_T2) * 100,
})
comparacao

## 3. Mapa de transições e tendência espacial

**Mapa de transições:** onde ocorreram as principais mudanças de classe.

**Tendência espacial:** como no LCM, ajustamos uma **superfície polinomial** ao mapa binário
(1 = virou a classe foco, 0 = não virou). Ordem 1 = um plano inclinado (direção geral);
ordens maiores = padrões mais complexos. O valor não é probabilidade: é a intensidade relativa da mudança.

In [ ]:
# Mapa das transições mais importantes (por área)
t1, t2 = uso[ANO_T1], uso[ANO_T2]
mudou = mascara & (t1 != t2)
pares = pd.Series({(a, b): MT_CAL.values[COD.index(a), COD.index(b)]
                   for a in COD for b in COD if a != b}).sort_values(ascending=False)
principais = [p for p in pares.index[:6] if pares[p] > 0]

mapa_trans = np.zeros(t1.shape, dtype=np.int16)
mapa_trans[mudou] = len(principais) + 1                      # "outras transições"
for k, (a, b) in enumerate(principais, start=1):
    mapa_trans[mudou & (t1 == a) & (t2 == b)] = k

cores_t = ['#f2f2f2', '#d4271e', '#e8a33d', '#2f6f9f', '#7b3294', '#1f8d49', '#a6611a', '#9e9e9e']
cores_t = cores_t[:len(principais) + 1] + ['#9e9e9e']
fig, ax = plt.subplots(figsize=(13, 9))
ax.imshow(np.where(mascara, mapa_trans, 0), cmap=ListedColormap(cores_t),
          vmin=-0.5, vmax=len(cores_t) - 0.5, interpolation='nearest')
ax.axis('off'); ax.set_title(f'Transições {ANO_T1} → {ANO_T2}')
leg = [Patch(color=cores_t[0], label='Sem mudança')]
leg += [Patch(color=cores_t[k], label=f'{CLASSES[a]} → {CLASSES[b]} ({pares[(a, b)]:,.0f} ha)')
        for k, (a, b) in enumerate(principais, start=1)]
leg += [Patch(color=cores_t[-1], label='Outras transições')]
ax.legend(handles=leg, loc='upper left', bbox_to_anchor=(1, 1), frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
def tendencia_espacial(binario, ordem):
    # Ajusta z = Σ a_ij · x^i · y^j (i + j ≤ ordem) por mínimos quadrados nos pixels da área de estudo
    yy, xx = np.mgrid[0:LINHAS, 0:COLUNAS]
    x = (xx / (COLUNAS - 1)) * 2 - 1                       # coordenadas normalizadas em [-1, 1]
    y = (yy / (LINHAS - 1)) * 2 - 1
    termos = [(i, j) for i in range(ordem + 1) for j in range(ordem + 1 - i)]
    A = np.column_stack([x[mascara] ** i * y[mascara] ** j for i, j in termos])
    coef, *_ = np.linalg.lstsq(A, binario[mascara].astype(float), rcond=None)
    sup = sum(c * x ** i * y ** j for c, (i, j) in zip(coef, termos))
    return sup

virou_foco = mascara & (t1 != CLASSE_FOCO) & (t2 == CLASSE_FOCO)
fig, axs = plt.subplots(1, 4, figsize=(20, 7))
axs[0].imshow(np.where(mascara, virou_foco, np.nan), cmap=ListedColormap(['#e9eef0', '#d4271e']), interpolation='nearest')
axs[0].set_title(f'Mudança para {CLASSES[CLASSE_FOCO]}\n(mapa binário)'); axs[0].axis('off')
for ax, ordem in zip(axs[1:], (1, 3, 5)):
    mostrar_continuo(tendencia_espacial(virou_foco, ordem), f'Tendência – polinômio de ordem {ordem}', ax, cmap='RdYlBu_r')
plt.tight_layout(); plt.show()

## 4. Seleção de variáveis: V de Cramér

Antes de treinar a rede, medimos a **associação** entre cada variável e a ocorrência da transição.
A variável contínua é dividida em faixas (quantis) e cruzada com "mudou / não mudou".

Leitura usual no LCM: **< 0,15 fraca · 0,15 a 0,40 útil · > 0,40 forte**.
Variáveis com V muito baixo podem ser retiradas do submodelo.

In [ ]:
def distancia_classe(mapa, classe):
    # Distância euclidiana (m) até o pixel mais próximo da classe (para a variável dinâmica)
    return ndimage.distance_transform_edt(mapa != classe, sampling=(DY, DX))

def matriz_variaveis(mapa_base):
    # Empilha as variáveis (linhas × colunas × n). A dinâmica é recalculada a partir de mapa_base.
    camadas = []
    for n in NOMES_VAR:
        if n == VAR_DINAMICA:
            camadas.append(distancia_classe(mapa_base, CLASSE_DINAMICA))
        else:
            camadas.append(variaveis[n])
    return np.stack(camadas, axis=-1)

def rotulos_submodelo(transicoes, t1, t2):
    # Rótulos de treino: cada transição do submodelo e a persistência de cada classe de origem
    origens = sorted({a for a, _ in transicoes})
    rot = np.full(t1.shape, '', dtype=object)
    for a in origens:
        rot[mascara & (t1 == a) & (t2 == a)] = f'Persiste {a}'
    for a, b in transicoes:
        rot[mascara & (t1 == a) & (t2 == b)] = f'{a}→{b}'
    return rot

def v_cramer(var, mudou, n_faixas=10):
    if len(var) < 20 or mudou.all() or not mudou.any():
        return np.nan
    faixas = pd.qcut(var, n_faixas, labels=False, duplicates='drop')
    tabela = pd.crosstab(faixas, mudou)
    if tabela.shape[1] < 2 or tabela.shape[0] < 2:
        return np.nan
    chi2 = chi2_contingency(tabela, correction=False)[0]
    return np.sqrt(chi2 / (tabela.values.sum() * (min(tabela.shape) - 1)))

if VAR_DINAMICA in variaveis:
    print(f'"{VAR_DINAMICA}" é dinâmica: recalculada como distância à classe {CLASSE_DINAMICA} de cada mapa base.')
X_T1 = matriz_variaveis(uso[ANO_T1])

# Conferência: quantos pixels cada transição dos submodelos tem entre T1 e T2
print(f'Pixels de cada transição dos submodelos ({ANO_T1} → {ANO_T2}):')
for nome_sm, trans in SUBMODELOS.items():
    for a, b in trans:
        n_px = np.sum(mascara & (uso[ANO_T1] == a) & (uso[ANO_T2] == b))
        aviso = '' if n_px >= 10 else '   ⚠ poucos ou nenhum pixel: será retirada'
        print(f'  {nome_sm:<22} {a:>3} → {b:<3} {n_px:>7} pixels{aviso}')

tab_v = {}
for nome_sm, trans in SUBMODELOS.items():
    rot = rotulos_submodelo(trans, uso[ANO_T1], uso[ANO_T2])
    validos = rot != ''
    mudou_sm = np.char.find(rot[validos].astype(str), '→') >= 0
    tab_v[nome_sm] = {n: v_cramer(X_T1[..., k][validos], mudou_sm) for k, n in enumerate(NOMES_VAR)}
tab_v = pd.DataFrame(tab_v)
display(tab_v.style.format('{:.3f}').background_gradient(cmap='Greens', vmin=0, vmax=0.5))

## 5. Potencial de transição com rede neural (MLP)

Para cada submodelo, treinamos uma **MLP** (Multi-Layer Perceptron), como no LCM:

- **Entradas:** as variáveis explicativas (padronizadas)
- **Saídas:** cada transição do submodelo + a persistência de cada classe de origem
- **Amostras:** até `AMOSTRAS_POR_CLASSE` pixels de cada saída; metade treina, metade testa. As saídas com poucos pixels são repetidas no treino para todas pesarem igual (equilíbrio, como a amostragem por classe do LCM)
- **Acurácia:** calculada de forma equilibrada (média dos acertos de cada saída), senão a persistência, muito mais numerosa, inflaria o valor
- **Skill measure:** acurácia descontado o acerto ao acaso → `S = (A − 1/K) / (1 − 1/K)`, com K = nº de saídas
- **Sensibilidade:** como no LCM, mantemos uma variável constante (na média) e vemos quanto a acurácia cai

O **potencial de transição** de cada pixel é a resposta da rede para aquela transição (0 a 1).

In [ ]:
def treinar_submodelo(nome, transicoes, t1, t2, X):
    rot = rotulos_submodelo(transicoes, t1, t2).ravel().astype(str)
    # Quantos pixels existem de cada saída (transição ou persistência)
    contagem = {r: int(np.sum(rot == r)) for r in [f'{a}→{b}' for a, b in transicoes] +
                [f'Persiste {a}' for a in sorted({a for a, _ in transicoes})]}
    vazias = [r for r, n in contagem.items() if n < 10]
    if vazias:
        print(f'\n⚠ {nome}: saídas com menos de 10 pixels foram retiradas: { {r: contagem[r] for r in vazias} }')
    validas = [r for r, n in contagem.items() if n >= 10]
    if not any('→' in r for r in validas) or not any(r.startswith('Persiste') for r in validas):
        print(f'⚠ {nome}: submodelo IGNORADO – não há pixels suficientes das transições {transicoes} '
              f'entre os mapas. Confira os códigos das classes em SUBMODELOS e a matriz de transição (Seção 2).')
        return None
    transicoes = [(a, b) for a, b in transicoes if f'{a}→{b}' in validas]
    idx_amostra = []
    for r in validas:
        idx = np.flatnonzero(rot == r)
        if idx.size > AMOSTRAS_POR_CLASSE:
            idx = rng.choice(idx, AMOSTRAS_POR_CLASSE, replace=False)
        idx_amostra.append(idx)
    idx_amostra = np.concatenate(idx_amostra)
    Xa = X.reshape(-1, X.shape[-1])[idx_amostra]
    # Rótulos em texto → números (versões recentes do scikit-learn exigem rótulos numéricos na MLP)
    nomes_saida, ya = np.unique(rot[idx_amostra], return_inverse=True)

    X_tr, X_te, y_tr, y_te = train_test_split(Xa, ya, test_size=0.5, stratify=ya, random_state=SEMENTE)
    # Equilíbrio entre as saídas: as transições costumam ter poucos pixels e a persistência, muitos.
    # Sem equilíbrio a rede aprende só "não muda". Repetimos (com reposição) as amostras das saídas menores.
    n_max = pd.Series(y_tr).value_counts().max()
    idx_eq = np.concatenate([rng.choice(np.flatnonzero(y_tr == r), n_max, replace=True) for r in np.unique(y_tr)])
    X_tr, y_tr = X_tr[idx_eq], y_tr[idx_eq]
    escala = StandardScaler().fit(X_tr)
    n_saidas = len(np.unique(ya))
    ocultos = max(4, int(round(np.sqrt(X.shape[-1] * n_saidas))) + 2)
    mlp = MLPClassifier(hidden_layer_sizes=(ocultos,), activation='logistic', solver='adam',
                        learning_rate_init=0.01, max_iter=1500, early_stopping=True,
                        validation_fraction=0.15, n_iter_no_change=50, random_state=SEMENTE)
    mlp.fit(escala.transform(X_tr), y_tr)

    # Acurácia equilibrada = média dos acertos de cada saída (igual a testar com o mesmo nº de pixels por saída)
    acerto = lambda X_: balanced_accuracy_score(y_te, mlp.predict(escala.transform(X_)))
    acc = acerto(X_te)
    skill = (acc - 1 / n_saidas) / (1 - 1 / n_saidas)
    sens = {}
    for k, n in enumerate(NOMES_VAR):
        X_c = X_te.copy(); X_c[:, k] = X_tr[:, k].mean()
        sens[n] = acc - acerto(X_c)

    print(f'\n■ {nome}: {n_saidas} saídas, {len(ya)} amostras, {ocultos} neurônios ocultos')
    print(f'  Contagem de amostras: { {str(n): int(c) for n, c in zip(nomes_saida, np.bincount(ya))} }')
    print(f'  Acurácia equilibrada (teste): {acc:.1%} | Skill measure: {skill:.3f}')
    return {'nome': nome, 'transicoes': transicoes, 'mlp': mlp, 'escala': escala, 'nomes_saida': list(nomes_saida),
            'acuracia': acc, 'skill': skill, 'sensibilidade': sens,
            'curva': mlp.loss_curve_, 'curva_val': mlp.validation_scores_}

def calcular_potenciais(modelos, mapa_base, X):
    # Potencial de cada transição (a, b): resposta da MLP nos pixels que estão hoje na classe a
    pot = {}
    Xf = X.reshape(-1, X.shape[-1])
    for mod in modelos:
        classes_saida = [mod['nomes_saida'][c] for c in mod['mlp'].classes_]
        origens = sorted({a for a, _ in mod['transicoes']})
        idx = np.flatnonzero((mascara & np.isin(mapa_base, origens)).ravel())
        prob = mod['mlp'].predict_proba(mod['escala'].transform(Xf[idx])) if idx.size else None
        for a, b in mod['transicoes']:
            p = np.zeros(mapa_base.size)
            rotulo = f'{a}→{b}'
            if idx.size and rotulo in classes_saida:
                da_origem = mapa_base.ravel()[idx] == a
                p[idx[da_origem]] = prob[da_origem, classes_saida.index(rotulo)]
            pot[(a, b)] = p.reshape(mapa_base.shape)
    return pot

modelos_cal = [treinar_submodelo(n, tr, uso[ANO_T1], uso[ANO_T2], X_T1) for n, tr in SUBMODELOS.items()]
modelos_cal = [m for m in modelos_cal if m is not None]
assert modelos_cal, 'Nenhum submodelo pôde ser treinado: confira SUBMODELOS e a matriz de transição.'

In [ ]:
# Desempenho e sensibilidade das variáveis
resumo = pd.DataFrame({m['nome']: {'Acurácia (%)': m['acuracia'] * 100, 'Skill': m['skill']} for m in modelos_cal}).T
display(resumo.style.format({'Acurácia (%)': '{:.1f}', 'Skill': '{:.3f}'}))

sens = pd.DataFrame({m['nome']: m['sensibilidade'] for m in modelos_cal}) * 100
print('Queda da acurácia (pontos %) ao manter cada variável constante – quanto maior, mais importante:')
display(sens.style.format('{:.1f}').background_gradient(cmap='Oranges'))

fig, axs = plt.subplots(1, len(modelos_cal), figsize=(5 * len(modelos_cal), 3.5), squeeze=False)
for ax, m in zip(axs[0], modelos_cal):
    ax.plot(m['curva'], label='erro de treino'); ax2 = ax.twinx()
    ax2.plot(m['curva_val'], color='#c46a2b', label='acurácia de validação')
    ax.set_title(m['nome']); ax.set_xlabel('iteração'); ax.set_ylabel('erro'); ax2.set_ylabel('acurácia')
plt.tight_layout(); plt.show()

In [ ]:
# Mapas de potencial de transição (calibração: base T2 = 2010, pronta para prever 2025)
X_T2 = matriz_variaveis(uso[ANO_T2])
POT_T2 = calcular_potenciais(modelos_cal, uso[ANO_T2], X_T2)

n = len(POT_T2)
fig, axs = plt.subplots(1, n, figsize=(5 * n, 6), squeeze=False)
for ax, ((a, b), p) in zip(axs[0], POT_T2.items()):
    mostrar_continuo(p, f'Potencial {CLASSES[a]} → {CLASSES[b]}', ax, cmap='magma', vmin=0, vmax=1)
plt.tight_layout(); plt.show()

## 6. Quantidade de mudança: Cadeia de Markov

A matriz de **probabilidades de transição** P vem da matriz de transição de T1→T2, dividindo cada linha pelo seu total.
Para projetar *n* anos com um intervalo de calibração de *Δ* anos, usamos **P^(n/Δ)**.

A demanda de cada transição modelada (em pixels) é: `área da classe de origem no mapa base × P[origem, destino]`.

In [ ]:
def markov(t1, t2, anos_calib, anos_prev):
    cont = matriz_transicao(t1, t2, unidade='px').values
    tot = cont.sum(axis=1, keepdims=True)
    P = np.divide(cont, tot, out=np.eye(len(COD)), where=tot > 0)
    Pn = np.real(fractional_matrix_power(P, anos_prev / anos_calib))
    Pn = np.clip(Pn, 0, None); Pn /= Pn.sum(axis=1, keepdims=True)
    return P, Pn

def demanda_transicoes(mapa_base, Pn, transicoes):
    n_base = {c: np.sum(mascara & (mapa_base == c)) for c in COD}
    return {(a, b): int(round(n_base[a] * Pn[COD.index(a), COD.index(b)])) for a, b in transicoes}

P_cal, Pn_cal = markov(uso[ANO_T1], uso[ANO_T2], ANO_T2 - ANO_T1, ANO_T3 - ANO_T2)
rotulos = [CLASSES[c] for c in COD]
print(f'Probabilidades de transição projetadas de {ANO_T2} para {ANO_T3}:')
display(pd.DataFrame(Pn_cal, index=rotulos, columns=rotulos).style.format('{:.3f}'))

TRANSICOES = [t for m in modelos_cal for t in m['transicoes']]   # só as transições modeladas
DEM_VAL = demanda_transicoes(uso[ANO_T2], Pn_cal, TRANSICOES)
display(pd.DataFrame({'pixels': list(DEM_VAL.values()), 'ha': [v * AREA_HA for v in DEM_VAL.values()]},
                     index=[f'{CLASSES[a]} → {CLASSES[b]}' for a, b in DEM_VAL]))

## 7. Predição *hard* e *soft*

**Hard (alocação competitiva):** todas as transições disputam os pixels. Ordenamos todos os pares (pixel, transição)
pelo potencial, do maior para o menor, e alocamos até cumprir a demanda de cada transição. Um pixel só muda uma vez.
Com variável dinâmica, a simulação é feita em `N_ETAPAS`: a cada etapa a distância à área urbana é recalculada
e os potenciais são atualizados (como as *recalculation stages* do LCM).

**Soft (vulnerabilidade):** combinação dos potenciais das transições para a classe foco pelo "OU lógico":
`V = 1 − Π(1 − p)`. Mostra onde a pressão é maior, **sem decidir** quais pixels mudam.

In [ ]:
def alocar(mapa, potenciais, demanda):
    novo = mapa.copy().ravel()
    lista_p, lista_px, lista_t = [], [], []
    chaves = [t for t in demanda if demanda[t] > 0]
    for k, (a, b) in enumerate(chaves):
        p = potenciais[(a, b)].ravel()
        idx = np.flatnonzero(mascara.ravel() & (mapa.ravel() == a) & (p > 0))
        lista_p.append(p[idx]); lista_px.append(idx); lista_t.append(np.full(idx.size, k))
    if not chaves:
        return mapa.copy()
    p, px, tt = np.concatenate(lista_p), np.concatenate(lista_px), np.concatenate(lista_t)
    ordem = np.argsort(-p, kind='stable')
    restante = np.array([demanda[t] for t in chaves])
    ocupado = np.zeros(novo.size, bool)
    for i in ordem:
        k, pix = tt[i], px[i]
        if restante[k] > 0 and not ocupado[pix]:
            novo[pix] = chaves[k][1]; ocupado[pix] = True; restante[k] -= 1
            if not restante.any():
                break
    return novo.reshape(mapa.shape)

def prever(modelos, mapa_base, demanda_total, n_etapas=N_ETAPAS):
    mapa = mapa_base.copy()
    etapas = n_etapas if VAR_DINAMICA in variaveis else 1
    for e in range(etapas):
        dem = {t: (v * (e + 1)) // etapas - (v * e) // etapas for t, v in demanda_total.items()}
        pot = calcular_potenciais(modelos, mapa, matriz_variaveis(mapa))
        mapa = alocar(mapa, pot, dem)
        print(f'  etapa {e + 1}/{etapas}: {sum(dem.values()):,} pixels alocados')
    return mapa

def vulnerabilidade(potenciais, classe_destino=CLASSE_FOCO):
    v = np.ones(mascara.shape)
    for (a, b), p in potenciais.items():
        if b == classe_destino:
            v *= (1 - p)
    return 1 - v

print(f'Predição hard {ANO_T2} → {ANO_T3}:')
PRED_T3 = prever(modelos_cal, uso[ANO_T2], DEM_VAL)
SOFT_T3 = vulnerabilidade(POT_T2)

fig, axs = plt.subplots(1, 3, figsize=(18, 7))
mostrar_uso(uso[ANO_T2], f'Real {ANO_T2} (base)', axs[0])
mostrar_uso(PRED_T3, f'Predição hard {ANO_T3}', axs[1])
mostrar_continuo(SOFT_T3, f'Predição soft: vulnerabilidade a {CLASSES[CLASSE_FOCO]}', axs[2], cmap='inferno', vmin=0, vmax=1)
fig.legend(handles=LEGENDA_USO, loc='lower center', ncol=5, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

## 8. Validação: o modelo acertou 2025?

Comparamos três mapas: **2010 real (base)**, **2025 previsto** e **2025 real**. Cada pixel cai em uma categoria:

| Categoria | Mudou na realidade? | Mudou na previsão? |
|---|---|---|
| **Acerto** | sim | sim, para a classe certa |
| **Acerto com classe errada** | sim | sim, para outra classe |
| **Omissão** | sim | não |
| **Falso alarme** | não | sim |
| **Persistência correta** | não | não |

**Figura de mérito** = acertos ÷ (acertos + classe errada + omissões + falsos alarmes). Valores baixos são comuns:
prever *onde* a mudança ocorre é difícil.

In [ ]:
def validar(base, previsto, real, titulo, classe=None):
    if classe is None:
        obs, prev = (real != base), (previsto != base)
        certo = previsto == real
    else:  # validação só da mudança PARA uma classe
        obs = (real == classe) & (base != classe)
        prev = (previsto == classe) & (base != classe)
        certo = np.ones_like(obs)
    m = mascara
    cat = np.zeros(base.shape, np.int8)                    # 0 persistência correta
    cat[m & obs & prev & certo] = 1                        # acerto
    cat[m & obs & prev & ~certo] = 2                       # classe errada
    cat[m & obs & ~prev] = 3                               # omissão
    cat[m & ~obs & prev] = 4                               # falso alarme
    cont = {k: np.sum(m & (cat == i)) for i, k in enumerate(
        ['Persistência correta', 'Acerto', 'Classe errada', 'Omissão', 'Falso alarme'])}
    fom = cont['Acerto'] / max(1, cont['Acerto'] + cont['Classe errada'] + cont['Omissão'] + cont['Falso alarme'])
    print(f'{titulo}: figura de mérito = {fom:.1%}')
    return cat, cont, fom

CAT, CONT, FOM = validar(uso[ANO_T2], PRED_T3, uso[ANO_T3], f'Todas as mudanças {ANO_T2}→{ANO_T3}')
CAT_F, CONT_F, FOM_F = validar(uso[ANO_T2], PRED_T3, uso[ANO_T3], f'Mudança para {CLASSES[CLASSE_FOCO]}', classe=CLASSE_FOCO)

display(pd.DataFrame({'Todas (ha)': {k: v * AREA_HA for k, v in CONT.items()},
                      f'Para {CLASSES[CLASSE_FOCO]} (ha)': {k: v * AREA_HA for k, v in CONT_F.items()}}))

cores_v = ['#eeeeee', '#1f8d49', '#e8a33d', '#2f6f9f', '#d4271e']
nomes_v = ['Persistência correta', 'Acerto', 'Classe errada', 'Omissão', 'Falso alarme']
fig, axs = plt.subplots(1, 2, figsize=(16, 8))
for ax, cat, tit in zip(axs, (CAT, CAT_F), (f'Todas as mudanças (FoM = {FOM:.1%})',
                                            f'Mudança para {CLASSES[CLASSE_FOCO]} (FoM = {FOM_F:.1%})')):
    ax.imshow(np.where(mascara, cat, np.nan), cmap=ListedColormap(cores_v), vmin=-0.5, vmax=4.5, interpolation='nearest')
    ax.set_title(tit); ax.axis('off')
fig.legend(handles=[Patch(color=c, label=n) for c, n in zip(cores_v, nomes_v)], loc='lower center', ncol=5, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

# Quantidade: previsto × real por classe
quant = pd.DataFrame({f'Real {ANO_T2}': [np.sum(mascara & (uso[ANO_T2] == c)) for c in COD],
                      f'Previsto {ANO_T3}': [np.sum(mascara & (PRED_T3 == c)) for c in COD],
                      f'Real {ANO_T3}': [np.sum(mascara & (uso[ANO_T3] == c)) for c in COD]},
                     index=rotulos) * AREA_HA
quant['Erro de quantidade'] = quant[f'Previsto {ANO_T3}'] - quant[f'Real {ANO_T3}']
quant

### A predição soft acerta onde a mudança aconteceu?

Se a vulnerabilidade for informativa, os pixels que **de fato** viraram a classe foco devem ter valores mais altos.
A curva ROC resume isso: AUC = 0,5 é acaso; 1,0 é perfeito.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
cand = mascara & (uso[ANO_T2] != CLASSE_FOCO)
alvo = (uso[ANO_T3] == CLASSE_FOCO)[cand]
if alvo.any() and (~alvo).any():
    auc = roc_auc_score(alvo, SOFT_T3[cand])
    fpr, tpr, _ = roc_curve(alvo, SOFT_T3[cand])
    plt.figure(figsize=(5, 5)); plt.plot(fpr, tpr, color='#d4271e', label=f'AUC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], 'k--', lw=0.8); plt.xlabel('Falsos positivos'); plt.ylabel('Verdadeiros positivos')
    plt.title(f'Vulnerabilidade × mudança real para {CLASSES[CLASSE_FOCO]} ({ANO_T2}–{ANO_T3})'); plt.legend()
    plt.tight_layout(); plt.show()
else:
    print('Não há mudança para a classe foco no período de validação.')

## 9. Cenário futuro

Recalibramos tudo com o período mais recente (**2010 → 2025**): novos potenciais (MLP), nova Cadeia de Markov,
e projetamos **2040** a partir do mapa real de 2025.

In [ ]:
print(f'Recalibrando com {ANO_T2} → {ANO_T3}...')
X_T2_cen = matriz_variaveis(uso[ANO_T2])
modelos_cen = [treinar_submodelo(n, tr, uso[ANO_T2], uso[ANO_T3], X_T2_cen) for n, tr in SUBMODELOS.items()]
modelos_cen = [m for m in modelos_cen if m is not None]
assert modelos_cen, 'Nenhum submodelo pôde ser treinado: confira SUBMODELOS e a matriz de transição.'

_, Pn_cen = markov(uso[ANO_T2], uso[ANO_T3], ANO_T3 - ANO_T2, ANO_CENARIO - ANO_T3)
DEM_CEN = demanda_transicoes(uso[ANO_T3], Pn_cen, [t for m in modelos_cen for t in m['transicoes']])
print(f'\nDemanda {ANO_T3} → {ANO_CENARIO} (ha):', {f'{a}→{b}': round(v * AREA_HA, 1) for (a, b), v in DEM_CEN.items()})

print(f'\nPredição hard {ANO_T3} → {ANO_CENARIO}:')
PRED_CEN = prever(modelos_cen, uso[ANO_T3], DEM_CEN)
POT_CEN = calcular_potenciais(modelos_cen, uso[ANO_T3], matriz_variaveis(uso[ANO_T3]))
SOFT_CEN = vulnerabilidade(POT_CEN)

fig, axs = plt.subplots(1, 3, figsize=(18, 7))
mostrar_uso(uso[ANO_T3], f'Real {ANO_T3}', axs[0])
mostrar_uso(PRED_CEN, f'Cenário hard {ANO_CENARIO}', axs[1])
mostrar_continuo(SOFT_CEN, f'Vulnerabilidade a {CLASSES[CLASSE_FOCO]} ({ANO_T3}→{ANO_CENARIO})', axs[2], cmap='inferno', vmin=0, vmax=1)
fig.legend(handles=LEGENDA_USO, loc='lower center', ncol=5, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

areas = pd.DataFrame({str(a): [np.sum(mascara & (mp == c)) for c in COD]
                      for a, mp in [(ANO_T1, uso[ANO_T1]), (ANO_T2, uso[ANO_T2]), (ANO_T3, uso[ANO_T3]), (ANO_CENARIO, PRED_CEN)]},
                     index=rotulos) * AREA_HA
display(areas)
areas.T.plot(marker='o', color=[CORES[c] for c in COD], figsize=(8, 4), title='Área por classe (ha) – 2040 é cenário')
plt.ylabel('ha'); plt.tight_layout(); plt.show()

## 10. Exportar resultados (`.rst` para o TerrSet)

Os mapas são salvos em `PASTA/resultados_python`, no mesmo formato e com o mesmo `.rdc` de referência
dos mapas de uso. Assim dá para abri-los no TerrSet e comparar com as saídas do LCM.

In [ ]:
SAIDA = os.path.join(PASTA, 'resultados_python')
os.makedirs(SAIDA, exist_ok=True)

def salvar_rst(nome, arr, legenda=None):
    modelo = open(meta_uso[ANO_T3]['arquivo'], encoding='latin-1').read().splitlines()
    inteiro = legenda is not None
    dados = np.where(mascara, arr, 0).astype(np.uint8 if inteiro else np.float32)
    dados.tofile(os.path.join(SAIDA, nome + '.rst'))
    linhas = []
    for l in modelo:
        chave = l.split(':')[0].strip()
        if chave.startswith('code'):
            continue
        if chave == 'file title':   l = f'file title  : {nome}'
        elif chave == 'data type':  l = f"data type   : {'byte' if inteiro else 'real'}"
        elif chave in ('min. value', 'display min'): l = f'{chave:<12}: {dados.min():.6g}'
        elif chave in ('max. value', 'display max'): l = f'{chave:<12}: {dados.max():.6g}'
        elif chave == 'legend cats':
            linhas.append(f'legend cats : {len(legenda) if inteiro else 0}')
            if inteiro:
                linhas += [f'code {c:>6} : {n}' for c, n in legenda.items()]
            continue
        linhas.append(l)
    with open(os.path.join(SAIDA, nome + '.rdc'), 'w', encoding='latin-1', newline='\r\n') as f:
        f.write('\n'.join(linhas) + '\n')

import unicodedata
leg_ascii = {c: unicodedata.normalize('NFKD', n).encode('ascii', 'ignore').decode() for c, n in CLASSES.items()}
salvar_rst(f'py_hard_{ANO_T3}', PRED_T3, leg_ascii)
salvar_rst(f'py_soft_{ANO_T3}', SOFT_T3)
salvar_rst(f'py_validacao_{ANO_T3}', CAT, {0: 'Persistencia correta', 1: 'Acerto', 2: 'Classe errada',
                                           3: 'Omissao', 4: 'Falso alarme'})
salvar_rst(f'py_hard_{ANO_CENARIO}', PRED_CEN, leg_ascii)
salvar_rst(f'py_soft_{ANO_CENARIO}', SOFT_CEN)
for (a, b), p in POT_T2.items():
    salvar_rst(f'py_potencial_{a}_{b}', p)

with pd.ExcelWriter(os.path.join(SAIDA, 'tabelas_mudanca.xlsx')) as xl:
    MT_CAL.to_excel(xl, sheet_name='Matriz transicao')
    GP_CAL.to_excel(xl, sheet_name='Ganhos e perdas')
    contrib.to_frame('ha').to_excel(xl, sheet_name='Contribuicoes')
    tab_v.to_excel(xl, sheet_name='V de Cramer')
    resumo.to_excel(xl, sheet_name='MLP')
    quant.to_excel(xl, sheet_name='Validacao quantidade')
    areas.to_excel(xl, sheet_name='Cenario')
print('Arquivos salvos em', SAIDA); print(sorted(os.listdir(SAIDA)))

## Para discutir: Python × LCM

| | TerrSet LCM | Este notebook |
|---|---|---|
| Matriz, ganhos, perdas, troca | Gráficos prontos | Cada fórmula visível |
| Potencial de transição | MLP (ou SimWeight, LR) | MLP do scikit-learn – dá para trocar por Random Forest em uma linha |
| Quantidade | Cadeia de Markov | Cadeia de Markov (mesma conta) |
| Alocação | MOLA (competitiva) | Alocação competitiva por ordenação do potencial |
| Reprodutibilidade | Cliques na interface | Código versionável e reexecutável |

**Perguntas:**
1. Os ganhos, perdas e a matriz batem com o Change Analysis do LCM?
2. As variáveis mais importantes (sensibilidade) são as mesmas nos dois?
3. A figura de mérito do Python ficou próxima da do LCM? Por que poderia ser diferente?
4. A tendência de 1995–2010 se manteve em 2010–2025? O que isso diz sobre o cenário de 2040?